# Desafio Sobrevivendo ao Titanic (Parte 1) — Pré-processamento e `Perceptron` Linear

Chegou o momento de integrar **Pandas + NumPy + Redes Neurais** em um problema real completo!
Temos dois arquivos em `../arquivos-csv/`:
- **`titanic_train.csv`:** Passageiros com desfecho conhecido (`survived` = `0` ou `1`) para treinar e validar o modelo.
- **`titanic_exercise.csv`:** Passageiros sem a coluna `survived`, para os quais devemos prever a sobrevivência e salvar em `.csv`.

---

## Como Preparar Dados Reais sem Vazamento (*Data Leakage*)?

Antes de alimentar qualquer neurônio artificial, precisamos realizar **4 etapas de Engenharia de Atributos**:
1. **Extração do Título Social (`Mr`, `Miss`, `Mrs`, `Master`):** No nome `"Allison, Master. Hudson Trevor"`, a palavra antes do ponto (`.`) indica se a pessoa era um menino pequeno (`Master`, ~5 anos) ou um adulto (`Mr`, ~30 anos) — muito mais preciso para estimar idades faltantes!
2. **Tamanho da Família:** `family = sibsp + parch + 1`.
3. **Imputação Calibrada APENAS no Treino:** Calculamos as medianas de idade e tarifa **somente em `titanic_train.csv`** e aplicamos esses mesmos valores fixos para tapar buracos em `titanic_exercise.csv` (evitando *Data Leakage*!).
4. **Padronização Z-score (`StandardScaler`):** Coloca todas as colunas na mesma ordem de grandeza (média `0` e desvio `1`), calibrando com `.fit_transform()` no treino e `.transform()` no teste.

---

## O Desafio (Parte 1 — Classificador Linear)
Implemente um **Perceptron Simples** para classificar os passageiros de `titanic_exercise.csv`, meça sua acurácia em uma divisão local de validação (`20%` do treino) e gere o arquivo de resposta `.csv`. Abaixo da sua célula de prática você encontra **dois gabaritos comentados linha a linha** (usando `Scikit-Learn` e `Python/NumPy Puro`).

In [ ]:
# Escreva sua implementação aqui!

<details>
<summary><b>Clique aqui para ver o Gabarito 1.1 — Perceptron Simples com Scikit-Learn (Comentado)</b></summary>

```python
import pandas as pd
from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# 1. Carregando conjuntos de treino e exercício (tratando '?' como ausente)
train_df = pd.read_csv("../arquivos-csv/titanic_train.csv", na_values=["?"])
test_df = pd.read_csv("../arquivos-csv/titanic_exercise.csv", na_values=["?"])

def extrair_atributos_basicos(df):
    '''Extrai colunas numéricas e título social sem calcular estatísticas ainda.'''
    X = pd.DataFrame()
    X["pclass"] = pd.to_numeric(df["pclass"], errors="coerce")
    X["is_male"] = df["sex"].map({"male": 1, "female": 0})
    X["age"] = pd.to_numeric(df["age"], errors="coerce")
    X["family"] = pd.to_numeric(df["sibsp"], errors="coerce") + pd.to_numeric(df["parch"], errors="coerce") + 1
    X["fare"] = pd.to_numeric(df["fare"], errors="coerce")

    # Expressão regular com raw string (r'...') extrai a palavra imediatamente antes do ponto '.'
    titulos = df["name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
    mapa_titulos = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4}
    X["title"] = titulos.map(mapa_titulos).fillna(5)
    return X

X_all = extrair_atributos_basicos(train_df)
y_all = train_df["survived"].to_numpy()
X_ex = extrair_atributos_basicos(test_df)

# 2. Imputação SEM vazamento de dados: aprendemos as medianas APENAS em X_all (treino)!
medianas_idade_por_titulo = X_all.groupby("title")["age"].median()
mediana_geral = X_all.median()

for tabela in (X_all, X_ex):
    # Preenche idade faltante usando a mediana do título correspondente aprendida no treino
    tabela["age"] = tabela["age"].fillna(tabela["title"].map(medianas_idade_por_titulo))
    # Qualquer NaN residual (ex.: fare) recebe a mediana geral do treino
    tabela.fillna(mediana_geral, inplace=True)

# 3. Validação honesta (80% treino interno / 20% validação) para medir acurácia real
X_tr, X_val, y_tr, y_val = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)

scaler_val = StandardScaler()
clf_val = Perceptron(max_iter=1000, random_state=42)
clf_val.fit(scaler_val.fit_transform(X_tr), y_tr)
acc_val = accuracy_score(y_val, clf_val.predict(scaler_val.transform(X_val))) * 100.0
print(f"Acurácia de validação do Perceptron (Scikit-Learn): {acc_val:.2f}%")

# 4. Treinamento final com 100% do treino e geração do CSV de resposta
scaler_final = StandardScaler()
X_all_scaled = scaler_final.fit_transform(X_all)
X_ex_scaled = scaler_final.transform(X_ex)

clf_final = Perceptron(max_iter=1000, random_state=42)
clf_final.fit(X_all_scaled, y_all)

y_pred_ex = clf_final.predict(X_ex_scaled)
pd.DataFrame({"id": test_df["id"], "survived": y_pred_ex}).to_csv("solution_perceptron_sklearn.csv", index=False)
print("Arquivo gerado com sucesso: solution_perceptron_sklearn.csv")
```

</details>

<details>
<summary><b>Clique aqui para ver o Gabarito 1.2 — Perceptron Simples em Python/NumPy Puro (Comentado)</b></summary>

```python
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1. Reaproveitamos o mesmo carregamento e limpeza sem Data Leakage
train_df = pd.read_csv("../arquivos-csv/titanic_train.csv", na_values=["?"])
test_df = pd.read_csv("../arquivos-csv/titanic_exercise.csv", na_values=["?"])

def preparar_sem_vazamento(df_treino, df_teste):
    def montar_base(df):
        X = pd.DataFrame()
        X["pclass"] = pd.to_numeric(df["pclass"], errors="coerce")
        X["is_male"] = df["sex"].map({"male": 1, "female": 0})
        X["age"] = pd.to_numeric(df["age"], errors="coerce")
        X["family"] = pd.to_numeric(df["sibsp"], errors="coerce") + pd.to_numeric(df["parch"], errors="coerce") + 1
        X["fare"] = pd.to_numeric(df["fare"], errors="coerce")
        titulos = df["name"].str.extract(r" ([A-Za-z]+)\.", expand=False)
        X["title"] = titulos.map({"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4}).fillna(5)
        return X

    X_tr = montar_base(df_treino)
    X_te = montar_base(df_teste)

    # Estatísticas calculadas exclusivamente no treino!
    med_titulo = X_tr.groupby("title")["age"].median()
    med_geral = X_tr.median()

    for X_tab in (X_tr, X_te):
        X_tab["age"] = X_tab["age"].fillna(X_tab["title"].map(med_titulo))
        X_tab.fillna(med_geral, inplace=True)

    scaler = StandardScaler()
    return scaler.fit_transform(X_tr), scaler.transform(X_te)

X_train_scaled, X_test_scaled = preparar_sem_vazamento(train_df, test_df)
y_train = train_df["survived"].to_numpy()

# 2. Treinamento manual do Perceptron (saída degrau 0 ou 1)
# Vetor w tem tamanho (n_features + 1), onde w[0] é o limiar associado a x0 = -1
w = np.zeros(X_train_scaled.shape[1] + 1)
taxa_aprendizado = 0.01
epocas = 100

for _ in range(epocas):
    for i in range(len(X_train_scaled)):
        # Insere a entrada fixa x0 = -1.0 na posição 0 da amostra
        x_amostra = np.insert(X_train_scaled[i], 0, -1.0)
        # Função degrau binária: 1 se u >= 0, senão 0
        y_hat = 1 if np.dot(w, x_amostra) >= 0.0 else 0
        # Atualização proporcional ao erro (d - y_hat)
        w = w + taxa_aprendizado * (y_train[i] - y_hat) * x_amostra

# 3. Previsão sobre os passageiros de titanic_exercise.csv
previsoes = [
    1 if np.dot(w, np.insert(linha, 0, -1.0)) >= 0.0 else 0
    for linha in X_test_scaled
]

pd.DataFrame({"id": test_df["id"], "survived": previsoes}).to_csv("solution_perceptron_puro.csv", index=False)
print("Arquivo gerado com sucesso: solution_perceptron_puro.csv")
```

</details>

---

## Próximo Passo
Como o naufrágio envolve interações não lineares (ex.: crianças da 2ª classe sobreviveram quase 100%, enquanto adultos homens da 2ª classe quase não sobreviveram!), uma única reta/hiperplano tem capacidade limitada. Avance para a **[Parte 2 — Perceptron Multicamadas (MLP)](05_desafio_titanic_parte2_mlp.ipynb)** para superar o modelo linear!